In [0]:
from pyspark.sql.functions import to_date, current_date, col

# Step-1: Read from Bronze (Streaming)
# 
df = spark.readStream.table("retails.bronze.orders_raw")


In [0]:
from pyspark.sql.functions import col

# Step 2: Separate Good vs Bad Records 
valid_condition = (
    (col("_rescued_data").isNull()) &
    (col("_corrupt_record").isNull())
)

invalid_df = df.filter(~valid_condition)
valid_df = df.filter(valid_condition)



In [0]:
from pyspark.sql.functions import to_json, struct, when, col, sha2, concat_ws

# Step 4: Apply Type Casting & Standardization
valid_df = valid_df.withColumn("order_id", col("order_id").cast("bigint")) \
                    .withColumn("order_date", col("order_date").cast("date")) \
                    .withColumn("order_customer_id", col("order_customer_id").cast(("bigint"))) \
                    .withColumn("order_status", col("order_status").cast("string")) \
                    .withColumn("batch_id", col("batch_id").cast("integer")) \
                    .withColumn("is_deleted", when(col("op")=='DELETE', True).otherwise(False))


In [0]:
# invalid_df.printSchema()

In [0]:
# Step 5: Data Quality Checks
valid_price_con = (
        col("order_id").isNotNull() &
        col("order_date").isNotNull() &
        col("order_customer_id").isNotNull() &
        col("order_status").isNotNull()
)
quality_valid_df = valid_df.filter(valid_price_con)

invalid_quality_df = valid_df.filter(~valid_price_con)
invalid_df = invalid_df.union(invalid_quality_df.drop("is_deleted"))
# invalid_quality_df.printSchema()


In [0]:
# Step-3: Drop raw/debug columns
# Keep _rescued_data only in Bronze for debugging
# 
quality_valid_df = quality_valid_df.drop("raw_data") \
        .drop("_rescued_data") \
        .drop("_corrupt_record") \
        .drop("source_file_path") \
        .drop("replay_flag")

In [0]:
from pyspark.sql.functions import current_timestamp, sha2, concat_ws

quality_valid_df = quality_valid_df \
            .withColumn("event_ts", current_timestamp()) \
            .withColumn("record_hash",
                        sha2(
                            concat_ws(
                                "||",
                                col("order_id"),
                                col("order_date"),
                                col("order_customer_id"),
                                col("order_status")
                            ),
                            256
                        )
                    )

In [0]:
# Step 6: Deduplication
silver_df = quality_valid_df.dropDuplicates(["record_hash","op", "run_id"])

# Step 7: Remove NA/None/NAN/null records for targeting spasific columns
silver_df = silver_df.dropna(how="any", subset=["order_date", "order_customer_id", "order_status"])

# Step 8: Fill 'Unknown' for NA/None/NAN/null for targeting spasific columns
# silver_df = silver_df.fillna("Unknown", subset=["product_description"])

In [0]:
from pyspark.sql.functions import *
import uuid

# Step 7: Quarantine table fill the metadata columns
quarantine_df = (
    invalid_df
  # Generate quarantine ID
    .withColumn("quarantine_id", expr("uuid()"))

    # -----------------------------------------
    # Error Reason
    # -----------------------------------------

    .withColumn(
        "error_reason",
        when(
            col("_corrupt_record").isNotNull(),
            lit("Malformed JSON record")
        ).when(
            col("_rescued_data").isNotNull(),
            lit("Schema drift / unexpected columns")
        )
    )

    # -----------------------------------------
    # Error Category
    # -----------------------------------------

    .withColumn(
        "error_category",
        when(
            col("_corrupt_record").isNotNull(),
            lit("CORRUPT_RECORD")
        ).when(
            col("_rescued_data").isNotNull(),
            lit("SCHEMA_DRIFT")
        )
    )

    # -----------------------------------------
    # Failed Column
    # -----------------------------------------

    .withColumn(
        "failed_column",
        when(
            col("_corrupt_record").isNotNull(),
            lit("FULL_RECORD")
        ).when(
            col("_rescued_data").isNotNull(),
            lit("_rescued_data")
        )
    )

    # -----------------------------------------
    # Validation Rule
    # -----------------------------------------

    .withColumn(
        "validation_rule",
        when(
            col("_corrupt_record").isNotNull(),
            lit("Valid JSON format expected")
        ).when(
            col("_rescued_data").isNotNull(),
            lit("Schema must match expected schema")
        )
    )

    # -----------------------------------------
    # Metadata Columns
    # -----------------------------------------

    .withColumn("quarantine_status", lit("NEW"))

    .withColumn("rejected_at", current_timestamp())

    .withColumn("reprocessed_at", lit(None).cast("timestamp"))

)

quarantine_stream_df = (
    quarantine_df
        .withColumn("order_id", col("order_id").cast("integer")) \
        .withColumn("order_customer_id", col("order_customer_id").cast("integer")) \
        .withColumn("batch_id", col("batch_id").cast("string"))
)

In [0]:
def process_quarantine_batch(batch_df, batch_id):
    # batch_df.createOrReplaceGlobalTempView("batch_df")
    try:
        batch_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "false") \
            .saveAsTable("retails.silver.orders_quarantine")
    except Exception as e:
        print(str(e))


In [0]:
# Step 8: Store Bad Records Separately (Quarantine Table)
try:
    quarantine_stream_df.writeStream \
        .foreachBatch(process_quarantine_batch) \
        .option("checkpointLocation", "dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_quarantine") \
        .trigger(availableNow=True) \
        .start()

except Exception as e:
    print(str(e))



In [0]:
def batch_append_to_cdc(batch_df, batch_id):
    try:
        batch_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "false") \
            .saveAsTable("retails.silver.orders_cdc")
    except Exception as e:
        raise e


In [0]:
#write append data in orders_cdc table
try:
    silver_df.writeStream \
        .foreachBatch(batch_append_to_cdc) \
        .option("checkpointLocation", "dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_cdc/") \
        .trigger(availableNow=True) \
        .start()
except Exception as e:
    print(str(e))


In [0]:
# %run ./orders_rescued_fix

In [0]:
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/bronze/orders/")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_quarantine/")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_cdc/")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_cleaned/")
# dbutils.fs.ls("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/orders/")

# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/bronze/orders/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_cdc/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_quarantine/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/silver/orders_cleaned/", True)

# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/orders/", True)
# dbutils.fs.rm("dbfs:/Volumes/data/raw/_checkpoints/retails/dev/gold/orders_clean/", True)



In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# # 1. Define the window specification
# window_spec = Window.partitionBy("product_id").orderBy(F.col("event_ts").desc())

# # 2. Apply row_number() and filter for ranks 1 and 2
# result_df = (
#     spark.table("retails.silver.orders_cdc")
#     .withColumn("rank", F.row_number().over(window_spec))
#     .filter(F.col("rank") == 2)
# )

# # 3. View the results
# display(result_df)


In [0]:
%sql
-- select * from retails.bronze.orders_raw;
-- select * from retails.silver.orders_cdc ;
-- select * from retails.silver.orders_quarantine;
-- select * from retails.silver.orders_cleaned
-- select * from retails.gold.dim_orders ;

-- truncate table retails.silver.orders_cleaned;
-- truncate table retails.bronze.orders_raw;
-- truncate table retails.silver.orders_quarantine;
-- truncate table retails.silver.orders_cdc;
-- truncate table retails.gold.dim_orders;